# Train Qwen3.5-0.8B as a decision model on title/abstract screening (Unsloth)

Same pipeline as [train_qwen35_4b.ipynb](train_qwen35_4b.ipynb) (data, leakage check, metrics, outputs). **Only the
model size changes** (`unsloth/Qwen3.5-4B` → `unsloth/Qwen3.5-0.8B`, same family and architecture), so the result
tells whether the gap between Laya and Qwen3.5-4B on val (FINDINGS entries 9 and 10) comes from model size.

| | |
|---|---|
| Model | `unsloth/Qwen3.5-0.8B` (pinned revision), with a **new decision head** trained together with LoRA adapters (r 16) on the frozen backbone |
| Data | balanced train requests, one random exclude per include (27,004 requests); val is the same file as for every other run |
| Schedule | as the 4B run: lr 2e-4 cosine **over 10 epochs** (head 1e-4), effective batch 64 (16 × 4), seed 3407; **stopped after 3 epochs** (1,266 steps) |
| Validation | step 0 and after every epoch; 17 CLEF TAR 2019 Intervention training reviews, abstract-level and content-level labels |
| GPU | **only physical GPU 1** |

**What the 4B run showed, and how this notebook uses it**

- **The 4B run plateaued after epoch 1–2.** WSS@95%, TNR@95% and AUC stayed within 0.5 points over epochs 1–3; MAP
  went 57.4 → 60.4 → 58.6 (abstract level). It was stopped after epoch 3. This run stops after 3 epochs too, but
  keeps the 4B schedule (cosine over 10 epochs, so the learning rate is still about 1.6e-4 at the stop), so both
  runs see exactly the same learning rate at every step and can be compared **epoch by epoch**. A schedule that
  decays fully within 3 epochs is a separate change; test it on its own if wanted.
- **Single reviews swing by about 6 MAP (SD) between epochs**, mostly the reviews with 1–5 includes. Compare runs
  per review with a bootstrap CI over reviews (last section), not by the mean alone, and treat the best of 3 epochs
  as optimistic.
- **Scores shift between epochs** (median exclude score 0.024 → 0.084 → 0.016 for 4B), so val loss and fixed
  thresholds do not track ranking quality. Model selection uses abstract-level MAP; compare ranking metrics.
- **The 4B run used the slow PyTorch path** for Qwen3.5's linear-attention layers: `flash-linear-attention` and
  `causal-conv1d` were installed while its kernel was already running. The first cell now checks that the fast
  kernels are in use and stops otherwise. The 4B validation pass took 37 min; expect far less here.
- **All checkpoints are kept** (3 × a small adapter), and training ends on its own, so the best checkpoint is loaded
  at the end (no manual interrupt).

**Step 0** is the untrained model: the head is new, so its ranking should be close to random. If step 0 or epoch 1
is clearly *below* random order (MAP about 6.5 / 4.7), the yes/no columns are swapped (`YES` below).

**Memory:** `BATCH` is the per-device batch; gradient accumulation keeps the effective batch at 64. Lower
`BATCH` (8, 4, ...) and `ACCUM` to match if training runs out of GPU memory, and restart the kernel.

Run headless on the server:
```bash
tmux new -s train
source .venv/bin/activate
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 \
    notebooks/training/train_qwen35_0.8b.ipynb --output-dir runs/notebooks
```

In [1]:
# GPU 1 only. Must run before torch / unsloth are imported (restart the kernel if they already are).
import os, sys
assert "torch" not in sys.modules, "Restart the kernel: torch was imported before the GPU was chosen."
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # number GPUs as nvidia-smi does
os.environ["CUDA_VISIBLE_DEVICES"] = "1"         # physical GPU 1; GPU 0 is never used

import unsloth  # noqa: F401  (Unsloth patches transformers, so it is imported first)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, "expected exactly one visible GPU (physical GPU 1)"
print("training on:", torch.cuda.get_device_name(0), "= physical GPU", os.environ["CUDA_VISIBLE_DEVICES"])


# Qwen3.5's linear-attention layers must use the fast kernels (flash-linear-attention, causal-conv1d). Transformers
# picks them when its Qwen3.5 module is first imported; if a package was installed after the kernel started, it
# silently uses the slow PyTorch reference (as in the 4B run). Restart the kernel after installing them.
from transformers.models.qwen3_5 import modeling_qwen3_5 as qwen35

def kernel_in_use(fn):
    cells = dict(zip(fn.__code__.co_freevars, (c.cell_contents for c in fn.__closure__ or ())))
    return cells.get("implementation")

for name in ("causal_conv1d_fn", "causal_conv1d_update", "torch_chunk_gated_delta_rule", "torch_recurrent_gated_delta_rule"):
    impl = kernel_in_use(getattr(qwen35, name))
    assert impl is not None, f"cannot tell which kernel {name} uses (transformers changed): check the warnings in training"
    print(f"{name:34} -> {impl.__module__}.{impl.__name__}")
    assert not impl.__module__.startswith("transformers."), \
        f"{name} uses the slow PyTorch reference: install flash-linear-attention and causal-conv1d, restart the kernel"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/joshua/decision-model-screener/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 22:31:38.084000 1167183 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1118 22:31:38.100000 1167183 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
training on: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition = physical GPU 1
causal_conv1d_fn                   -> causal_conv1d.causal_conv1d_interface.causal_conv1d_fn
causal_conv1d_update               -> causal_conv1d.causal_conv1d_interface.causal_conv1d_update
torch_chunk_gated_delta_rule       -> fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule
torch_recurrent_gated_delta_rule   -> fla.ops.gated_delta_rule.fused_recurrent.fused_recurrent_gated_delta_rule


## Settings

In [ ]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
DATA = ROOT / "data" / "processed" / "training"
BALANCED = True    # True: train on 1 exclude per include (build with --train-neg-per-pos 1); val is unchanged
REQUESTS = "review-criteria__record-title+abstract__label-abstract_else_final" + ("__train-neg1" if BALANCED else "")   # output of scripts/build_requests.py
REQ_DIR = DATA / "requests" / REQUESTS

# Pinned Hub revisions (never an unpinned "latest" for a reported result)
MODELS = {"convaiinnovations/laya-multilingual": "92d22143468d8af669dc1a5a0480a2b21b4fe56e",
          "unsloth/Qwen3.5-0.8B": "23c69c53358a07516b5827588b3fdb12ae78fd65",
          "unsloth/Qwen3.5-4B": "3764fa359b9082ea5a1e4a5e3ac3aaf6e9671636"}
MODEL = "unsloth/Qwen3.5-0.8B"
REVISION = MODELS[MODEL]

# Everything below is as in the 4B run, so model size is the only difference
MAX_SEQ_LENGTH = 8192    # tokens; checked below: no val input is cut, train cuts are reported
LORA_R, LORA_ALPHA = 16, 16
LR, HEAD_LR = 2e-4, 1e-4  # LoRA adapters; the decision head (Unsloth's default)
TRAIN_HEAD = True         # Qwen: the decision head is new and must train (the notebook enforces this for Qwen anyway)
BATCH = 64                # per device; lower it (and raise ACCUM) if training runs out of GPU memory
ACCUM = 1#64 // BATCH      # effective batch 64
EPOCHS = 10               # length of the cosine schedule, as in the 4B run (one epoch: 422 steps)
EVAL_BATCH = 128          # lower it if the evaluation runs out of GPU memory
SEED = 3407

RUN = ROOT / "runs" / f"{MODEL.split('/')[-1]}__{REQUESTS}"
COMPARE_RUN = ROOT / "runs" / f"Qwen3.5-4B__{REQUESTS}"   # compared epoch by epoch in the last section
JEV_SCORES = ROOT / "results" / f"requests__{REQUESTS.removesuffix('__train-neg1')}__val" / "scores.jsonl"
assert not (RUN / "val_metrics.jsonl").exists(), f"{RUN} already has results: move them away or resume instead"
RUN.mkdir(parents=True, exist_ok=True)
print(RUN)

/home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1


## Data and leakage check

Train and val must not share a review, a review title or a paper (PMID, DOI or title, the same rule as the
build script). The check runs on the files that are actually trained on.

In [3]:
import json
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "scripts"))
from build_requests import paper_keys, norm_title   # same matching rule as the build
from screening import metrics                        # same metric code as the Jev experiments

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

build_config = json.loads((REQ_DIR / "config.json").read_text())
train_rows, train_meta = read_jsonl(REQ_DIR / "train.jsonl"), read_jsonl(REQ_DIR / "train.meta.jsonl")
val_rows, val_meta = read_jsonl(REQ_DIR / "val.jsonl"), read_jsonl(REQ_DIR / "val.meta.jsonl")
reviews = {r["review_id"]: r for r in read_jsonl(DATA / "reviews.jsonl")}

needed = {(m["review_id"], m["record_id"]) for m in train_meta + val_meta}
records = {}
with open(DATA / "records.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if (r["review_id"], r["record_id"]) in needed:
            records[r["review_id"], r["record_id"]] = r

train_reviews = {m["review_id"] for m in train_meta}
val_reviews = {m["review_id"] for m in val_meta}
assert not train_reviews & val_reviews, "review in train and val"
assert not ({norm_title(reviews[r]["title"]) for r in train_reviews}
            & {norm_title(reviews[r]["title"]) for r in val_reviews}), "same review title in train and val"
train_keys = set()
for m in train_meta:
    train_keys |= paper_keys(records[m["review_id"], m["record_id"]])
shared = [m for m in val_meta if paper_keys(records[m["review_id"], m["record_id"]]) & train_keys]
assert not shared, f"{len(shared)} val records are the same paper as a train record"
print(f"no overlap: {len(train_reviews)} train reviews / {len(train_rows):,} requests, "
      f"{len(val_reviews)} val reviews / {len(val_rows):,} requests")
print("train reviews dropped by the build for overlap:", build_config["train_reviews_dropped_for_overlap"])
print("train records removed as same paper as val/test:", build_config["train_records_dropped_same_paper_as_val_test"])

# Both CLEF TAR label levels for the val metrics (the request file carries the abstract-level one)
val_labels = {lvl: [records[m["review_id"], m["record_id"]][f"label_{lvl}"] for m in val_meta]
              for lvl in ("abstract", "final")}
del records

no overlap: 92 train reviews / 27,004 requests, 17 val reviews / 27,870 requests
train reviews dropped by the build for overlap: {'Brouwer_2019': '21% of CD010558', 'Kerschbaumer_2024a': 'same review title as Kerschbaumer_2024b', 'Parodis_2023a': 'same review title as Parodis_2023b', 'Webers_2022': '40% of Kerschbaumer_2024b'}
train records removed as same paper as val/test: 6431


In [4]:
# Request format (label on the question) -> Unsloth format (answers in "gold")
def to_unsloth(row):
    questions = {name: {k: v for k, v in q.items() if k != "label"} for name, q in row["questions"].items()}
    return {"state": row["state"], "questions": questions,
            "gold": {name: q["label"] for name, q in row["questions"].items()}}

train_data = [to_unsloth(r) for r in train_rows]
val_data = [to_unsloth(r) for r in val_rows]
print(json.dumps(val_data[0], indent=1, ensure_ascii=False)[:1500])

{
 "state": {
  "review": {
   "criteria": "Types of studies: We included only RCTs in this review.\nTypes of participants: We included trials in which participants had neovascular AMD as defined by study investigators.\nTypes of interventions: We included studies that compared anti‐VEGF treatment versus another treatment, sham treatment, or no treatment. We did not include studies that compared different doses of one anti‐VEGF treatment against another, studies that included no control or comparator group, or studies that used anti‐VEGF agents in combination with other treatments. We did not include studies of aflibercept (VEGF Trap‐Eye/EYLEA solution) or studies that compared different treatment schedules (e.g. monthly vs as needed dosing), because other Cochrane reviews have evaluated these interventions (Li 2016; Sarwar 2016).\nTypes of outcome measures: Primary outcomes The primary outcome for this review was based on best‐corrected visual acuity (BCVA) at one‐year follow‐up. All 

## Model: LoRA adapters only

In [5]:
from unsloth import FastDecisionModel, DecisionTrainer

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=MODEL,
    revision=REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,     # 16-bit backbone, bf16 like Laya's own training
    load_in_4bit=False,
    random_state=SEED,
)
model = FastDecisionModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    random_state=SEED,
)

# The backbone is model.encoder (Laya's mmBERT or Qwen); everything else is the decision head.
head_trains = TRAIN_HEAD or getattr(model, "is_clef", False)   # a Qwen head is new and must train
for n, p in model.named_parameters():
    if not n.startswith("encoder."):
        p.requires_grad_(head_trains)
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
backbone = [n for n in trainable if n.startswith("encoder.") and "lora_" not in n]
assert not backbone, f"backbone weights would train: {backbone[:5]}"
assert any("lora_" in n for n in trainable), "no LoRA adapters are trainable"
print("head trains:", head_trains)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"trainable {n_train / 1e6:.1f}M of {n_all / 1e6:.0f}M parameters")

# Position of the "yes" option in the logits: Laya orders noul options (false, true), Clef-style heads (true, false)
YES = 0 if getattr(model, "is_clef", False) else 1
print("max_len", model.decision_config["max_len"], "| yes option at index", YES)

==((====))==  Unsloth 2026.11.6: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition. Num GPUs = 1. Max memory: 94.968 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 6540.27it/s]


Unsloth: compiling each transformer block as one region (Qwen3_5DecoderLayer).


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 3116.12it/s]


head trains: True
trainable 38.1M of 891M parameters
max_len 8192 | yes option at index 0


In [6]:
train_items, train_report = FastDecisionModel.build_dataset(train_data, tokenizer, model)
val_items, val_report = FastDecisionModel.build_dataset(val_data, tokenizer, model)
print("train:", train_report)
print("val:  ", val_report)

# Never drop records; no val input may be cut (its abstract would be scored on less text).
assert train_report["skipped"] == 0 and val_report["skipped"] == 0, "records were skipped"
assert [it["row"] for it in val_items] == list(range(len(val_rows))), "val order changed"
assert val_report["truncated"] == 0, f"{val_report['truncated']} val inputs exceed MAX_SEQ_LENGTH: raise it"
lengths = sorted(len(it["input_ids"]) for it in val_items)
print("val tokens: median", lengths[len(lengths) // 2], "max", lengths[-1])

Unsloth: 1 of 27,004 training inputs are longer than max_seq_length = 8192, so the end of their state is cut. Raise max_seq_length to train on all of it.
train: {'total': 27004, 'skipped': 0, 'reason': None, 'truncated': 1}
val:   {'total': 27870, 'skipped': 0, 'reason': None, 'truncated': 0}
val tokens: median 1043 max 3719


## Validation metrics at each evaluation

Ranking per review by P(include), ties broken by record id. Reviews without includes at a label level are skipped
for that level (as in the Jev experiments).

In [7]:
import numpy as np
import pandas as pd

def ranking_metrics(p_include):
    df = pd.DataFrame({"review_id": [m["review_id"] for m in val_meta],
                       "record_id": [m["record_id"] for m in val_meta],
                       "p": p_include, **{f"label_{k}": v for k, v in val_labels.items()}})
    out = {}
    for level in ("abstract", "final"):
        per_review = []
        for _, g in df.groupby("review_id"):
            ranked = g.sort_values(["p", "record_id"], ascending=[False, True])
            m = metrics(ranked[f"label_{level}"].astype(int))
            if m:
                per_review.append(m)
        mean = pd.DataFrame(per_review).mean() * 100
        for name, key in (("MAP", "map"), ("WSS@95%", "wss95"), ("TNR@95%", "tnr95"), ("R@10%", "r10")):
            out[f"{key}_{level}"] = round(float(mean[name]), 2)
        out[f"reviews_{level}"] = len(per_review)
    return df, out

trainer = None   # set below; compute_metrics reads the current step from it

def compute_metrics(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    logits = torch.tensor(np.asarray(logits)[:, :2], dtype=torch.float32)
    p_include = torch.softmax(logits, -1)[:, YES].numpy()
    assert len(p_include) == len(val_meta)
    df, out = ranking_metrics(p_include)
    step = trainer.state.global_step if trainer is not None else 0
    (RUN / "val_scores").mkdir(parents=True, exist_ok=True)   # here, not at definition: RUN is set in the settings cell
    df.to_json(RUN / "val_scores" / f"step-{step:06d}.jsonl", orient="records", lines=True)
    with open(RUN / "val_metrics.jsonl", "a") as f:
        f.write(json.dumps({"step": step, **out}) + "\n")
    return out

## Training

In [8]:
from transformers import TrainingArguments, TrainerCallback
import subprocess

class StopAfterEpochs(TrainerCallback):
    """Stop after STOP_AFTER_EPOCHS epochs. Set at the end of an epoch, so that epoch is still evaluated and saved,
    and load_best_model_at_end still runs. The learning-rate schedule keeps its 10-epoch length."""
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch >= STOP_AFTER_EPOCHS - 1e-6:
            control.should_training_stop = True
        return control

args = TrainingArguments(
    output_dir=str(RUN / "checkpoints"),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    per_device_eval_batch_size=EVAL_BATCH,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",              # validation after every epoch (step 0 is evaluated below)
    save_strategy="epoch",              # one checkpoint (adapters + head) per evaluation
    save_total_limit=None,              # keep every epoch (the 4B run lost its epoch-1 checkpoint to the limit)
    load_best_model_at_end=True,
    metric_for_best_model="map_abstract",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
)
trainer = DecisionTrainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=train_items,
    eval_dataset=val_items,
    compute_metrics=compute_metrics,
    head_learning_rate=HEAD_LR,
    callbacks=[StopAfterEpochs()],
)

# What this run used, next to its results
(RUN / "run_config.json").write_text(json.dumps({
    "model": MODEL, "revision": REVISION, "requests": REQUESTS, "build_config": build_config,
    "max_seq_length": MAX_SEQ_LENGTH, "train_head": head_trains, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR, "head_lr": HEAD_LR,
    "batch": BATCH, "accum": ACCUM, "epochs": EPOCHS, "stop_after_epochs": STOP_AFTER_EPOCHS, "eval": "step 0 + every epoch",
    "balanced": BALANCED, "seed": SEED, "gpu": "physical GPU " + os.environ["CUDA_VISIBLE_DEVICES"],
    "train_report": train_report, "val_report": val_report}, indent=2, default=str))
(RUN / "pip_freeze.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                                   capture_output=True, text=True).stdout)

# Step 0: the untrained model on val (writes val_scores/step-000000.jsonl and a row in val_metrics.jsonl).
# Skip it when resuming, it is already on disk.
#step0 = trainer.evaluate()
#print({k: v for k, v in step0.items() if k.startswith("eval_map")}, f"| {step0['eval_runtime'] / 60:.1f} min per validation pass")

# Resume after an interruption: trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
print(train_result)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,004 | Num Epochs = 10 | Total steps = 4,220
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 38,147,588 of 891,133,508 (4.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss,Map Abstract,Wss95 Abstract,Tnr95 Abstract,R10 Abstract,Reviews Abstract,Map Final,Wss95 Final,Tnr95 Final,R10 Final,Reviews Final
1,0.583678,0.529926,15.570000,43.150000,50.080000,39.770000,17,11.400000,51.880000,58.210000,41.190000,16
2,0.471413,0.253213,19.640000,50.940000,57.980000,48.850000,17,17.030000,61.090000,67.630000,53.050000,16
3,0.372410,0.101029,21.380000,55.720000,63.070000,49.230000,17,15.900000,60.950000,67.570000,52.980000,16


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-422/.unsloth-save-dc4fwp0f/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-844/.unsloth-save-qfp7hjzq/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-1266/.unsloth-save-dtwvm71k/tokenizer_config.json.


TrainOutput(global_step=1266, training_loss=0.5281785956302902, metrics={'train_runtime': 5610.8387, 'train_samples_per_second': 48.128, 'train_steps_per_second': 0.752, 'total_flos': 0.0, 'train_loss': 0.5281785956302902, 'epoch': 3.0})


## Result

In [9]:
history = pd.read_json(RUN / "val_metrics.jsonl", lines=True).drop_duplicates("step", keep="last").set_index("step")
print("best checkpoint:", trainer.state.best_model_checkpoint)
history

best checkpoint: /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-1266


,map_abstract,wss95_abstract,tnr95_abstract,r10_abstract,reviews_abstract,map_final,wss95_final,tnr95_final,r10_final,reviews_final
step,,,,,,,,,,
422,15.57,43.15,50.08,39.77,17,11.40,51.88,58.21,41.19,16
844,19.64,50.94,57.98,48.85,17,17.03,61.09,67.63,53.05,16
1266,21.38,55.72,63.07,49.23,17,15.90,60.95,67.57,52.98,16


## Per review: against zero-shot Jev and Qwen3.5-4B at the same epoch

Same records and metric code as the training loop. Jev is ranked as in FINDINGS (ties broken by BM25, then record
id); trained models by score, then record id. AUC counts tied scores as half. Differences are means over reviews,
with a 95% bootstrap CI over reviews and the number of reviews better / worse. Reviews without includes at a label
level are skipped for that level. Epochs are compared at the same step, because both runs use the same schedule.

In [10]:
def per_review(df, col, level, tie=None):
    rows = {}
    for rid, g in df.groupby("review_id"):
        keys = [col] + ([tie] if tie else []) + ["record_id"]
        g = g.sort_values(keys, ascending=[False] * (len(keys) - 1) + [True])
        y = g[f"label_{level}"].to_numpy().astype(int)
        m = metrics(y)
        if not m:
            continue
        r = pd.Series(g[col].to_numpy()).rank().to_numpy()
        n1, n0 = y.sum(), len(y) - y.sum()
        rows[rid] = {**m, "AUC": (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)}
    return pd.DataFrame(rows).T * 100

rng = np.random.default_rng(0)
def diff(a, b):
    d = (a - b).dropna().to_numpy()
    lo, hi = np.percentile(rng.choice(d, (5000, len(d))).mean(1), [2.5, 97.5])
    return f"{d.mean():+.1f} [{lo:+.1f}, {hi:+.1f}] {(d > 0).sum()}/{(d < 0).sum()}"

def scores(run, step):
    path = run / "val_scores" / f"step-{step:06d}.jsonl"
    return pd.read_json(path, lines=True) if path.exists() else None

jev = pd.read_json(JEV_SCORES, lines=True)
steps = sorted(int(p.stem.split("-")[1]) for p in (RUN / "val_scores").glob("step-*.jsonl"))
summary, diffs = [], []
for level in ("abstract", "final"):
    J = per_review(jev, "jev", level, "bm25")
    summary.append({"level": level, "model": "Jev zero-shot", "step": "", **J.mean().round(1).to_dict()})
    for step in steps:
        small, big = scores(RUN, step), scores(COMPARE_RUN, step)
        S = per_review(small, "p", level)
        summary.append({"level": level, "model": "Qwen3.5-0.8B", "step": step, **S.mean().round(1).to_dict()})
        row = {"level": level, "step": step, "ΔMAP vs Jev": diff(S["MAP"], J["MAP"]),
               "ΔWSS@95% vs Jev": diff(S["WSS@95%"], J["WSS@95%"])}
        if big is not None:
            B = per_review(big, "p", level)
            summary.append({"level": level, "model": "Qwen3.5-4B", "step": step, **B.mean().round(1).to_dict()})
            row |= {"ΔMAP vs 4B": diff(S["MAP"], B["MAP"]), "ΔWSS@95% vs 4B": diff(S["WSS@95%"], B["WSS@95%"])}
        diffs.append(row)

cols = ["level", "model", "step", "MAP", "R@10%", "WSS@95%", "TNR@95%", "AUC"]
display(pd.DataFrame(summary)[cols])
display(pd.DataFrame(diffs))

,level,model,step,MAP,R@10%,WSS@95%,TNR@95%,AUC
0,abstract,Jev zero-shot,,55.1,81.5,75.2,83.9,95.6
1,abstract,Qwen3.5-0.8B,422,15.6,39.8,43.2,50.1,80.5
2,abstract,Qwen3.5-4B,422,57.4,84.1,77.0,85.9,96.6
3,abstract,Qwen3.5-0.8B,844,19.6,48.8,50.9,58.0,84.2
4,abstract,Qwen3.5-4B,844,60.4,84.9,76.8,85.8,96.6
5,abstract,Qwen3.5-0.8B,1266,21.4,49.2,55.7,63.1,85.2
6,abstract,Qwen3.5-4B,1266,58.5,83.9,76.7,85.7,96.8
7,final,Jev zero-shot,,59.5,95.4,85.1,92.8,98.2
8,final,Qwen3.5-0.8B,422,11.4,41.2,51.9,58.2,82.3
9,final,Qwen3.5-4B,422,57.3,91.3,83.9,91.6,97.7


,level,step,ΔMAP vs Jev,ΔWSS@95% vs Jev,ΔMAP vs 4B,ΔWSS@95% vs 4B
0,abstract,422,"-39.5 [-51.0, -27.5] 1/16","-32.0 [-42.3, -21.7] 1/16","-41.8 [-51.4, -32.0] 0/17","-33.8 [-43.1, -24.8] 0/17"
1,abstract,844,"-35.4 [-48.5, -21.9] 2/15","-24.2 [-36.6, -11.6] 2/15","-40.7 [-51.3, -29.6] 1/16","-25.8 [-37.4, -14.8] 1/16"
2,abstract,1266,"-33.7 [-47.9, -18.4] 3/14","-19.5 [-28.5, -9.5] 3/14","-37.2 [-47.1, -26.7] 1/16","-21.0 [-29.6, -13.4] 1/16"
3,final,422,"-48.1 [-60.7, -35.0] 0/16","-33.2 [-43.5, -22.8] 0/16","-45.9 [-56.4, -34.7] 0/16","-32.0 [-41.7, -23.1] 0/16"
4,final,844,"-42.5 [-57.2, -26.6] 3/13","-24.0 [-34.6, -13.1] 2/14","-42.5 [-55.6, -29.2] 1/15","-21.0 [-30.6, -11.9] 2/14"
5,final,1266,"-43.6 [-58.2, -27.6] 2/14","-24.1 [-34.3, -13.4] 2/14","-39.9 [-52.1, -27.2] 1/15","-21.1 [-30.5, -12.2] 2/14"


In [11]:
# MAP per review at the best epoch of each run (abstract level), next to Jev
best_small = int(history["map_abstract"].drop(0, errors="ignore").idxmax())
big_hist = pd.read_json(COMPARE_RUN / "val_metrics.jsonl", lines=True).drop_duplicates("step", keep="last").set_index("step")
best_big = int(big_hist["map_abstract"].drop(0, errors="ignore").idxmax())
level = "abstract"
table = pd.DataFrame({
    "includes": jev[jev[f"label_{level}"] == 1].groupby("review_id").size(),
    f"0.8B (step {best_small})": per_review(scores(RUN, best_small), "p", level)["MAP"],
    f"4B (step {best_big})": per_review(scores(COMPARE_RUN, best_big), "p", level)["MAP"],
    "Jev": per_review(jev, "jev", level, "bm25")["MAP"],
}).round(1)
table

,includes,0.8B (step 1266),4B (step 844),Jev
CD005139,112,5.9,42.9,16.7
CD005253,4,6.9,71.5,67.1
CD006715,13,16.5,76.2,84.8
CD007868,5,6.3,87.7,82.6
CD008018,17,19.8,71.7,75.0
CD008170,88,1.9,14.2,17.6
CD010019,1,2.9,50.0,50.0
CD010355,9,72.8,93.8,72.8
CD010526,21,22.9,78.6,73.0
CD010778,26,21.4,80.4,64.4


In [12]:
# Best model (loaded at the end of training): adapters + decision head only
model.save_pretrained(str(RUN / "best_adapter"))
tokenizer.save_pretrained(str(RUN / "best_adapter"))
print("saved", RUN / "best_adapter")

Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/best_adapter/.unsloth-save-2_i41xyk/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/best_adapter/tokenizer_config.json.


saved /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/best_adapter


In [13]:
# Continue the 0.8B run to the end of its 10-epoch schedule (step 4,220)
STOP_AFTER_EPOCHS = 10   # the StopAfterEpochs callback reads this value at every epoch end
train_result = trainer.train(resume_from_checkpoint=str(RUN / "checkpoints" / "checkpoint-1266"))
print(train_result)

history = pd.read_json(RUN / "val_metrics.jsonl", lines=True).drop_duplicates("step", keep="last").set_index("step")
print("best checkpoint:", trainer.state.best_model_checkpoint)
model.save_pretrained(str(RUN / "best_adapter")); tokenizer.save_pretrained(str(RUN / "best_adapter"))
history


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,004 | Num Epochs = 10 | Total steps = 4,220
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 38,147,588 of 891,133,508 (4.28% trained)


Epoch,Training Loss,Validation Loss,Map Abstract,Wss95 Abstract,Tnr95 Abstract,R10 Abstract,Reviews Abstract,Map Final,Wss95 Final,Tnr95 Final,R10 Final,Reviews Final
4,0.247385,0.143654,19.150000,53.910000,61.130000,48.480000,17,14.580000,60.530000,67.060000,51.220000,16
5,0.212546,0.091023,19.660000,42.700000,49.670000,48.510000,17,16.600000,51.610000,58.000000,53.340000,16
6,0.112605,0.197086,20.350000,42.590000,49.390000,44.040000,17,16.730000,52.340000,58.660000,45.410000,16
7,0.027510,0.391758,20.410000,39.590000,46.280000,44.720000,17,16.820000,46.490000,52.700000,49.530000,16
8,0.001001,0.581601,19.350000,29.230000,34.940000,42.610000,17,16.250000,35.270000,41.090000,48.370000,16
9,0.000019,0.555928,18.620000,24.980000,30.630000,42.470000,17,15.760000,32.680000,38.380000,48.260000,16
10,0.000010,0.557406,18.520000,25.410000,31.080000,41.950000,17,15.610000,31.900000,37.580000,47.490000,16


Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-1688/.unsloth-save-00fz9gy_/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-2110/.unsloth-save-z5r9003t/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-2532/.unsloth-save-yeiv6f1r/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-2954/.unsloth-s

TrainOutput(global_step=4220, training_loss=0.057753728873326964, metrics={'train_runtime': 13028.2055, 'train_samples_per_second': 20.727, 'train_steps_per_second': 0.324, 'total_flos': 0.0, 'train_loss': 0.057753728873326964, 'epoch': 10.0})
best checkpoint: /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-1266


Unsloth: Restored added_tokens_decoder metadata in /home/joshua/decision-model-screener/runs/Qwen3.5-0.8B__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/best_adapter/.unsloth-save-wlzzpvrr/tokenizer_config.json.


,map_abstract,wss95_abstract,tnr95_abstract,r10_abstract,reviews_abstract,map_final,wss95_final,tnr95_final,r10_final,reviews_final
step,,,,,,,,,,
422,15.57,43.15,50.08,39.77,17,11.40,51.88,58.21,41.19,16
844,19.64,50.94,57.98,48.85,17,17.03,61.09,67.63,53.05,16
1266,21.38,55.72,63.07,49.23,17,15.90,60.95,67.57,52.98,16
1688,19.15,53.91,61.13,48.48,17,14.58,60.53,67.06,51.22,16
2110,19.66,42.70,49.67,48.51,17,16.60,51.61,58.00,53.34,16
2532,20.35,42.59,49.39,44.04,17,16.73,52.34,58.66,45.41,16
2954,20.41,39.59,46.28,44.72,17,16.82,46.49,52.70,49.53,16
3376,19.35,29.23,34.94,42.61,17,16.25,35.27,41.09,48.37,16
3798,18.62,24.98,30.63,42.47,17,15.76,32.68,38.38,48.26,16
